In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# <a href="https://en.wikipedia.org/wiki/Unification_(computer_science)">Unification</a>

This notebook implements the algorithm of *Martelli and Montanari* for the unification of terms.

## Utility Functions

Terms are represented as values of the type `term` that is defined in the notebook `FOL-Parser.ipynb`:
```
type term =
  | Var of string                  (* variable *)
  | Fun of string * term list      (* function application, constants have no arguments *)
```
This notebook also provides the function `parse_term` that converts a string into a term, and the function `term_to_string` that converts a term into a string.  Our parser distinguishes variables and function symbols as follows:
- A word starting with an upper case letter is interpreted as a *variable*.
- A word starting with a lower case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
import_notebook "FOL-Parser.ipynb"

In [ ]:
parse_term "f(g(X),Y)"

A *substitution* $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$ maps variables to terms.  It is represented as a value of the type `substitution`, which is an abbreviation of the type `(string, term) RMap.t`.

The function `substitution_to_string` takes one argument:
- `sigma` is a substitution.

It returns a string representing `sigma`, e.g. `{X ↦ g(Z), Y ↦ h(U, V)}`.  `Fun.id` is the identity function, which converts the variables, which are strings already.

In [ ]:
type substitution = (string, term) RMap.t

let substitution_to_string (sigma : substitution) : string =
  RMap.to_string Fun.id term_to_string sigma

The function `apply_term` takes two arguments:
- `sigma` is a substitution,
- `t` is a term.

It computes the term $t\sigma$, i.e. it *applies* the substitution $\sigma$ to $t$.  Assume that $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$.  Then $t\sigma$ is defined by induction on $t$:
- If $t$ is a variable, there are two cases when defining $t\sigma$:
  - $t = x_i$ for an $i\in\{1,\cdots,n\}$.  Then we define
    $$ x_i\sigma := t_i. $$
  - $t = y$ where $y\in\mathcal{V}$, but $y \not\in \{x_1,\cdots,x_n\}$. Then we define
    $$ y\sigma := y.$$
- Otherwise, we must have $t = f(s_1,\cdots,s_m)$. Then we define:
  $$ f(s_1, \cdots, s_m)\sigma := f(s_1\sigma, \cdots, s_m\sigma). $$

In [ ]:
let rec apply_term (sigma : substitution) (t : term) : term =
  match t with
  | Var x ->
      (match RMap.find_opt x sigma with
       | Some s -> s
       | None   -> t)
  | Fun (f, ts) -> Fun (f, List.map (apply_term sigma) ts)

A *syntactical equation* $s \doteq t$ is represented as the pair `(s, t)` of type `equation`.

The function `apply_equation` takes two arguments:
- `sigma` is a substitution,
- `(s, t)` is a syntactical equation.

It applies `sigma` to both sides of the equation.  The function `apply_equations` takes a substitution `sigma` and a list of syntactical equations `es` and applies `sigma` to all equations of `es`.

In [ ]:
type equation = term * term   (* s ≐ t *)

let apply_equation (sigma : substitution) ((s, t) : equation) : equation =
  (apply_term sigma s, apply_term sigma t)

let apply_equations (sigma : substitution) (es : equation list) : equation list =
  List.map (apply_equation sigma) es

In [ ]:
let s1 = parse_term "g(Z)"
let s2 = parse_term "h(U, V)"
let sigma : substitution = RMap.of_list [("X", s1); ("Y", s2)]

In [ ]:
substitution_to_string sigma

In [ ]:
let t = parse_term "f(X,h(Y,X),g(Z))"

In [ ]:
term_to_string (apply_term sigma t)

If  $\sigma = \big\{ x_1 \mapsto s_1, \cdots, x_m \mapsto s_m \big\}$ and
$\tau = \big\{ y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$
are two substitutions that are <em style="color:blue;">non-overlapping</em>, i.e. such that $\texttt{dom}(\sigma) \cap \texttt{dom}(\tau) = \{\}$ holds,
then we define the <em style="color:blue;">composition</em> $\sigma\tau$ of $\sigma$ and $\tau$ as follows:
$$\sigma\tau := \big\{ x_1 \mapsto s_1\tau, \cdots, x_m \mapsto s_m\tau,\; y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$$
The function `compose` takes two arguments:
- `sigma` is a substitution,
- `tau` is a substitution such that `sigma` and `tau` are non-overlapping.

It returns the composition $\sigma\tau$.

**Implementation:** `RMap.map (apply_term tau) sigma` applies `tau` to all terms of `sigma`.  Then `RMap.fold` adds the pairs of `tau` one by one.

In [ ]:
let compose (sigma : substitution) (tau : substitution) : substitution =
  RMap.fold RMap.add tau (RMap.map (apply_term tau) sigma)

In [ ]:
let tau : substitution = RMap.of_list [("Z", s1); ("U", s2)]

In [ ]:
substitution_to_string sigma, substitution_to_string tau

In [ ]:
substitution_to_string (compose sigma tau)

The function `occurs` takes two arguments:
- `x` is a variable,
- `t` is a term.

It returns `true` if the variable $x$ occurs in the term $t$.

In [ ]:
let rec occurs (x : string) (t : term) : bool =
  match t with
  | Var y       -> x = y
  | Fun (_, ts) -> List.exists (occurs x) ts

In [ ]:
term_to_string t, occurs "U" t, occurs "X" t

## The Algorithm of Martelli and Montanari

The rules of Martelli and Montanari that can be used to solve a system of syntactical equations are as follows:
<ol>
<li> If $y\in\mathcal{V}$ is a variable that does <b style="color:red;">not</b> occur in the term $t$,
     then we perform the following reduction:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E\{y \mapsto t\}, \sigma\big\{ y \mapsto t \big\} \Big\rangle
     $$
</li>
<li> If the variable $y$ occurs in the term $t$, then the system of syntactical equations
     $E \cup \big\{ y \doteq t \big\}$ is not solvable:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle\;\leadsto\; \Omega \quad
        \mbox{if $y \in \textrm{Var}(t)$ and $y \not=t$.}$$
</li>
<li> If $y\in\mathcal{V}$ is a variable and $t$ is no variable, then we use the following rule:
     $$ \Big\langle E \cup \big\{ t \doteq y \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle.
     $$
</li>
<li> Trivial syntactical equations of variables can be dropped:
     $$ \Big\langle E \cup \big\{ x \doteq x \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E, \sigma \Big\rangle.
     $$
</li>
<li> If $f$ is an $n$-ary function symbol, then we have:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_n) \doteq f(t_1,\cdots,t_n) \big\}, \sigma \Big\rangle
         \;\leadsto\;
         \Big\langle E \cup \big\{ s_1 \doteq t_1, \cdots, s_n \doteq t_n\}, \sigma \Big\rangle.
     $$
</li>
<li> The system of syntactical equations $E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\}$
     has <b style="color:red;">no</b> solution if the function symbols $f$ and $g$ are different:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\},
      \sigma \Big\rangle \;\leadsto\; \Omega \qquad \mbox{if $f \not= g$}.
     $$
</ol>

The function `solve` takes two arguments:
- `es` is a list of <em style="color:blue;">syntactical equations</em>,
- `sigma` is a substitution.

It uses the rules of Martelli and Montanari to solve the equations `es` recursively.  The result has the type `substitution option`: If the equations are solvable, the result is `Some mu`, where `mu` is the composition of `sigma` with the most general unifier of `es`.  Otherwise, the result is `None`, which corresponds to $\Omega$ in the rules given above.

**Implementation:** Every recursive call of `solve` performs one step of the algorithm.  We take the first equation $s \doteq t$ from `es` and apply the rule that matches this equation.  The pattern `(s, t) :: rest when s = t` matches an equation whose sides are identical.  This covers rule 4, but also drops other trivial equations like $f(x) \doteq f(x)$.  The pattern `(s, (Var _ as t)) :: rest` matches an equation whose right side is a variable and binds this variable to `t`; as the previous pattern has not matched, `s` is no variable.  The function `List.combine` combines the arguments of two function applications into a list of pairs, i.e. into new equations.

In [ ]:
let rec solve (es : equation list) (sigma : substitution) : substitution option =
  match es with
  | [] -> Some sigma
  | (s, t) :: rest when s = t ->                              (* rule 4 *)
      solve rest sigma
  | (Var x, t) :: rest ->
      if occurs x t then
        None                                                  (* rule 2 *)
      else
        let tau = RMap.singleton x t in                       (* rule 1 *)
        solve (apply_equations tau rest) (compose sigma tau)
  | (s, (Var _ as t)) :: rest ->                              (* rule 3 *)
      solve ((t, s) :: rest) sigma
  | (Fun (f, ss), Fun (g, ts)) :: rest ->
      if f <> g || List.length ss <> List.length ts then
        None                                                  (* rule 6 *)
      else
        solve (List.combine ss ts @ rest) sigma               (* rule 5 *)

The function `unify` takes two arguments:
- `s` is a term,
- `t` is a term.

It returns the <em style="color:blue;">most general unifier</em> of $s$ and $t$ as `Some mu`, or `None` if $s$ and $t$ are not unifiable.

In [ ]:
let unify (s : term) (t : term) : substitution option =
  solve [(s, t)] RMap.empty

In [ ]:
let t1 = parse_term "p(X1,f(X1))"
let t2 = parse_term "p(g(X2),X3)"

In [ ]:
term_to_string t1, term_to_string t2

In [ ]:
let mu = unify t1 t2

In [ ]:
Option.map substitution_to_string mu

Let us check that $\mu$ is indeed a unifier, i.e. that $t_1\mu = t_2\mu$.  `Option.get` extracts the substitution from `Some mu`.

In [ ]:
let m = Option.get mu in
(term_to_string (apply_term m t1), term_to_string (apply_term m t2))

The terms $X$ and $f(X)$ are not unifiable:

In [ ]:
unify (parse_term "X") (parse_term "f(X)")